Análisis de Ventas E-commerce

Objetivo

Analizar el comportamiento de clientes, pedidos, productos, pagos y vendedores mediante la integración de múltiples tablas de una plataforma de comercio electrónico.

librerias a utilizar 

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12,6)

Carga de datos

In [2]:
customers = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\olist_customers_dataset.csv")

orders = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\olist_orders_dataset.csv")

order_items = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\olist_order_items_dataset.csv")

products = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\olist_products_dataset.csv")

payments = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\olist_order_payments_dataset.csv")

sellers = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\olist_sellers_dataset.csv")

categories = pd.read_csv(r"D:\proyectos\analisis-ventas-ecommerce\data\product_category_name_translation.csv")

Exploración inicial de las tablas

Antes de integrar los datos, se analizará la estructura de cada tabla para identificar sus claves y relaciones.

In [4]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)
print("Payments:", payments.shape)
print("Sellers:", sellers.shape)
print("Categories:", categories.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Products: (32951, 9)
Payments: (103886, 5)
Sellers: (3095, 4)
Categories: (71, 2)


Estructura de las tablas

Se analizan las columnas de cada tabla para identificar las claves primarias, claves foráneas y las relaciones necesarias para integrar los datos.

In [8]:
print("Customers:", customers.columns.tolist())
print("Orders:", orders.columns.tolist())
print("Order Items:", order_items.columns.tolist())
print("Products:", products.columns.tolist())
print("Payments:", payments.columns.tolist())
print("Sellers:", sellers.columns.tolist())

Customers: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Orders: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Order Items: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
Products: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Payments: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
Sellers: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


Identificación de relaciones entre tablas

Relaciones identificadas

customers se relaciona con orders mediante customer_id.
orders se relaciona con order_items mediante order_id.
order_items se relaciona con products mediante product_id.
order_items se relaciona con sellers mediante seller_id.
orders se relaciona con payments mediante order_id.

In [9]:
print("customer_id" in customers.columns)
print("customer_id" in orders.columns)

print("order_id" in orders.columns)
print("order_id" in order_items.columns)

print("product_id" in order_items.columns)
print("product_id" in products.columns)

print("seller_id" in order_items.columns)
print("seller_id" in sellers.columns)

True
True
True
True
True
True
True
True


Validación de relaciones

Integración de datos

En esta etapa se realizarán las uniones entre las tablas para construir un conjunto de datos integrado que facilite el análisis de negocio.

In [10]:
# Integración de todas las tablas

ecommerce = (
    orders
    .merge(customers, on="customer_id", how="inner")
    .merge(order_items, on="order_id", how="inner")
    .merge(products, on="product_id", how="inner")
    .merge(payments, on="order_id", how="inner")
    .merge(sellers, on="seller_id", how="inner")
)

print("Dimensiones de la tabla integrada:", ecommerce.shape)

ecommerce.head()

Dimensiones de la tabla integrada: (117601, 33)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,19.0,8.0,13.0,1,credit_card,1,18.12,9350,maua,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,19.0,8.0,13.0,3,voucher,1,2.00,9350,maua,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,19.0,8.0,13.0,2,voucher,1,18.59,9350,maua,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,19.0,13.0,19.0,1,boleto,1,141.46,31570,belo horizonte,SP
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,24.0,19.0,21.0,1,credit_card,3,179.12,14840,guariba,SP


se descarga como archivo csv la nueva tabla integrada

In [17]:
ecommerce.to_csv(
    "../data/ecommerce_final.csv",
    index=False
)

In [11]:
ecommerce.info()

<class 'pandas.DataFrame'>
RangeIndex: 117601 entries, 0 to 117600
Data columns (total 33 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       117601 non-null  str    
 1   customer_id                    117601 non-null  str    
 2   order_status                   117601 non-null  str    
 3   order_purchase_timestamp       117601 non-null  str    
 4   order_approved_at              117586 non-null  str    
 5   order_delivered_carrier_date   116356 non-null  str    
 6   order_delivered_customer_date  115034 non-null  str    
 7   order_estimated_delivery_date  117601 non-null  str    
 8   customer_unique_id             117601 non-null  str    
 9   customer_zip_code_prefix       117601 non-null  int64  
 10  customer_city                  117601 non-null  str    
 11  customer_state                 117601 non-null  str    
 12  order_item_id                  117601 non

Calidad de datos

Se analizarán los valores faltantes presentes en la tabla integrada para identificar posibles problemas de calidad en los datos.

In [15]:
ecommerce.isnull().sum().sort_values(ascending=False)

order_delivered_customer_date    2567
product_photos_qty               1698
product_description_lenght       1698
product_name_lenght              1698
product_category_name            1698
order_delivered_carrier_date     1245
product_width_cm                   20
product_weight_g                   20
product_length_cm                  20
product_height_cm                  20
order_approved_at                  15
order_id                            0
customer_id                         0
order_purchase_timestamp            0
order_status                        0
order_estimated_delivery_date       0
customer_unique_id                  0
price                               0
shipping_limit_date                 0
seller_id                           0
product_id                          0
order_item_id                       0
customer_state                      0
customer_city                       0
customer_zip_code_prefix            0
freight_value                       0
payment_sequ

Análisis de valores faltantes

Se identificaron valores nulos principalmente en variables relacionadas con la entrega de pedidos y características de algunos productos.

Los mayores niveles de ausencia se observan en:

order_delivered_customer_date
product_category_name
product_name_lenght
product_description_lenght
product_photos_qty
La presencia de valores faltantes en las fechas de entrega puede estar asociada a pedidos cancelados o aún no entregados al momento del registro de los datos.

Por otro lado, la ausencia de información en algunas características de productos sugiere problemas de calidad o registros incompletos dentro del catálogo.

En general, las variables clave para el análisis comercial como clientes, pedidos, precios, pagos y vendedores presentan una completitud prácticamente total.

Análisis Comercial

¿Cuál fue la facturación total del negocio y qué categorías generan más ingresos?

In [4]:

ecommerce = pd.read_csv(
    r"D:\proyectos\analisis-ventas-ecommerce\data\ecommerce_final.csv"
)

In [5]:
ecommerce["revenue"] = (
    ecommerce["price"] +
    ecommerce["freight_value"]
)

facturacion_total = ecommerce["revenue"].sum()
print(f"Facturación total: {facturacion_total:,.2f}")

Facturación total: 16,566,543.85


La facturación total registrada en la plataforma asciende a aproximadamente R$ 16,57 millones, considerando el valor de los productos y los costos de envío asociados a cada pedido.

Este resultado proporciona una visión general del volumen económico gestionado por el negocio durante el período analizado y servirá como referencia para evaluar el desempeño de categorías de productos, vendedores y regiones geográficas.

A partir de esta métrica es posible identificar qué segmentos del negocio generan la mayor contribución a los ingresos totales.

¿Qué categorías generan más ingresos?